# VEGFR2 Docking Pipeline — Results Exploration

Template notebook for exploring docking results.
Run `python pipeline.py run-all` before using this notebook.

Key output files:
- `data/results/docking_summary.csv` — best scores + interaction summary
- `data/results/*_scores.csv` — per-ligand, per-pose scores
- `data/results/plots/` — pre-generated plots

In [ ]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

results_dir = Path('..') / config['paths']['results']
print('Results directory:', results_dir)

In [ ]:
# Load docking summary
summary_path = results_dir / 'docking_summary.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
else:
    print('docking_summary.csv not found — run pipeline first')

In [ ]:
# Load all per-ligand scores
score_files = list(results_dir.glob('*_scores.csv'))
all_scores = pd.concat([pd.read_csv(f) for f in score_files], ignore_index=True)
print(f'Loaded {len(all_scores)} poses from {len(score_files)} ligands')
all_scores.head(10)

In [ ]:
# Score vs rank plot
fig, ax = plt.subplots(figsize=(9, 5))
for (ligand, backend), grp in all_scores.groupby(['ligand', 'backend']):
    grp_sorted = grp.sort_values('pose')
    ax.plot(grp_sorted['pose'], grp_sorted['score'],
            marker='o', label=f'{ligand} ({backend})')
ax.set_xlabel('Pose rank')
ax.set_ylabel('Score')
ax.set_title('Score vs pose rank')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Display pre-generated interaction heatmap
from IPython.display import Image
heatmap = results_dir / 'plots' / 'interaction_heatmap.png'
if heatmap.exists():
    display(Image(str(heatmap)))
else:
    print('Heatmap not yet generated (run Stage 5 with ProLIF installed)')